# Descoberta Orientada por Dados (Clusterização de Tópicos)
Este notebook cumpre a etapa metodológica da Parte 2 (Método A). Utilizaremos o **Latent Dirichlet Allocation (LDA)**, um modelo probabilístico que descobre tópicos ocultos na base de comentários do NPS.


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Carregando os dados
df = pd.read_excel(r"../../../../data/raw/dados.xlsx")
df = df.rename(columns={'Data Avaliação': 'data', 'CentroNv2': 'loja', 'Comentario': 'comentario'})
df_com_texto = df[(df['comentario'].notnull()) & (df['comentario'].str.strip() != '-')].copy()


## 1. Pré-processamento e Filtro
Como definido na EDA, removeremos comentários curtos (< 5 palavras) pois prejudicam a clusterização com ruídos.


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text) # Remove emojis
    text = re.sub(r'[^a-záéíóúâêôãõç\s]', '', text) # Remove pontuação
    text = re.sub(r'(.)\1{2,}', r'\1', text) # Remove spam
    return text.strip()

df_com_texto['comentario_limpo'] = df_com_texto['comentario'].apply(clean_text)
df_com_texto['qtd_palavras'] = df_com_texto['comentario_limpo'].apply(lambda x: len(x.split()))

# Filtro de ruído
df_model = df_com_texto[df_com_texto['qtd_palavras'] >= 5].copy()
print(f"Total de comentários aptos para modelagem: {len(df_model)}")


## 2. Vetorização e Stopwords
Removemos stopwords clássicas e algumas palavras específicas do negócio (ex: 'swift') que aparecem em todos os tópicos e não diferenciam temas.


In [ ]:
stopwords_pt = ['que', 'o', 'a', 'de', 'e', 'do', 'da', 'em', 'um', 'para', 'é', 'com', 'não', 'uma', 'os', 'no', 'se', 'na', 'por', 'mais', 'as', 'dos', 'como', 'mas', 'foi', 'ao', 'ele', 'das', 'tem', 'à', 'seu', 'sua', 'ou', 'ser', 'quando', 'muito', 'há', 'nos', 'já', 'está', 'eu', 'também', 'só', 'pelo', 'pela', 'até', 'isso', 'ela', 'entre', 'era', 'depois', 'sem', 'mesmo', 'aos', 'ter', 'seus', 'quem', 'nas', 'me', 'esse', 'eles', 'estão', 'você', 'tinha', 'foram', 'essa', 'num', 'nem', 'suas', 'meu', 'às', 'minha', 'têm', 'numa', 'pelos', 'elas', 'havia', 'seja', 'qual', 'será', 'nós', 'tenho', 'lhe', 'deles', 'essas', 'esses', 'pelas', 'este', 'fosse', 'dele', 'loja', 'swift', 'tudo', 'produtos', 'sempre', 'atendimento', 'carne', 'carnes', 'produto']

# Amostragem para agilizar o tempo de treinamento computacional
df_lda = df_model.sample(min(30000, len(df_model)), random_state=42)
corpus = df_lda['comentario_limpo'].tolist()

vectorizer = CountVectorizer(stop_words=stopwords_pt, max_df=0.90, min_df=5)
X = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()

def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Tópico #{topic_idx}: "
        message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)
    print()


## 3. Treinamento LDA e Análise de Tópicos
Vamos testar algoritmos probabilísticos para inferir tópicos com **K=5** e **K=10**.


In [ ]:
print("--- LDA com K=5 ---")
lda_5 = LatentDirichletAllocation(n_components=5, random_state=42, n_jobs=-1)
lda_5.fit(X)
print_top_words(lda_5, feature_names, 10)

print("\n--- LDA com K=10 ---")
lda_10 = LatentDirichletAllocation(n_components=10, random_state=42, n_jobs=-1)
lda_10.fit(X)
print_top_words(lda_10, feature_names, 10)


**Conclusão Metodológica do LDA:**
* O modelo com K=5 gerou tópicos muito genéricos. O K=10 apresentou excelente separação granular (ex: separando atendimento de caixa da qualidade da picanha/frango).
* **Ruído Descartado:** O Tópico #0 do K=10 (bem, bom, excelente) foi descartado por representar apenas sentimento generalista sem aspecto claro.
* **Colapso:** Colapsamos o Tópico #6 e #7 num grande tópico "Atendimento", e #4 e #5 num tópico "Qualidade do Produto".
* Com isso, derivamos a nossa Tabela de 5 Categorias Finais para a Entrega 2.
